In [1]:
import pandas as pd
from pathlib import Path

raw = Path("../data/raw")
tables = {f.stem: pd.read_csv(f) for f in sorted(raw.glob("*.csv"))}

orders = tables["olist_orders_dataset"]
customers = tables["olist_customers_dataset"]
items = tables["olist_order_items_dataset"]
payments = tables["olist_order_payments_dataset"]
sellers = tables["olist_sellers_dataset"]
geo = tables["olist_geolocation_dataset"]

In [2]:
items_total = (items.assign(total=items.price + items.freight_value)
                    .groupby("order_id").total.sum().rename("items_total"))
pay_total = payments.groupby("order_id").payment_value.sum().rename("pay_total")

cmp = pd.concat([items_total, pay_total], axis=1)
cmp["diff"] = (cmp.pay_total - cmp.items_total).round(2)
both = cmp.dropna()

print("Orders in both:", len(both))
print("Match (|diff| <= 0.01):", (both["diff"].abs() <= 0.01).sum())
print("Mismatch:", (both["diff"].abs() > 0.01).sum())
print("Items but no payment:", cmp.pay_total.isna().sum())
print("Payment but no items:", cmp.items_total.isna().sum())

Orders in both: 98665
Match (|diff| <= 0.01): 98362
Mismatch: 303
Items but no payment: 1
Payment but no items: 775


In [3]:
geo_zips = set(geo.geolocation_zip_code_prefix)

for label, df, col in [("customers", customers, "customer_zip_code_prefix"),
                       ("sellers", sellers, "seller_zip_code_prefix")]:
    missing = ~df[col].isin(geo_zips)
    print(f"{label}: {missing.sum()} rows without geolocation ({df.loc[missing, col].nunique()} distinct zips)")

print("Distinct zip prefixes in geolocation:", len(geo_zips))

customers: 278 rows without geolocation (157 distinct zips)
sellers: 7 rows without geolocation (7 distinct zips)
Distinct zip prefixes in geolocation: 19015


In [6]:
chg = customers.groupby("customer_unique_id")[
    ["customer_zip_code_prefix", "customer_city", "customer_state"]].nunique()

print("Customers with more than 1 zip prefix:", (chg.customer_zip_code_prefix > 1).sum())
print("Customers with more than 1 city:", (chg.customer_city > 1).sum())
print("Customers with more than 1 state:", (chg.customer_state > 1).sum())
mm = both[both["diff"].abs() > 0.01].copy()
print(mm["diff"].describe())
print("Paid MORE than items:", (mm["diff"] > 0).sum(), "| Paid LESS:", (mm["diff"] < 0).sum())

pt = payments.groupby("order_id").agg(
    n_pay=("payment_sequential", "count"),
    types=("payment_type", lambda s: ",".join(sorted(set(s)))),
    max_inst=("payment_installments", "max"),
)
mm = mm.join(pt)
print(mm.types.value_counts().head(5))
print(mm.max_inst.describe())

Customers with more than 1 zip prefix: 250
Customers with more than 1 city: 122
Customers with more than 1 state: 39
count    303.000000
mean       9.475446
std       18.077227
min      -51.620000
25%        1.355000
50%        5.410000
75%       11.830000
max      182.810000
Name: diff, dtype: float64
Paid MORE than items: 264 | Paid LESS: 39
types
credit_card            275
boleto                  14
credit_card,voucher      7
debit_card               7
Name: count, dtype: int64
count    303.000000
mean       5.858086
std        3.642509
min        1.000000
25%        3.000000
50%        5.000000
75%        8.000000
max       24.000000
Name: max_inst, dtype: float64
